In [1]:
!pip install groq


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from groq import Groq
import pymysql
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Establish connection using PyMySQL
conn = pymysql.connect(
    host="127.0.0.1",  # Using IP explicitly avoids localhost/IPv6 name resolution quirks
    port=3306,
    user="root",
    password="admin",
    database="crm_ai"
)

cursor = conn.cursor()
print("Connected successfully!")

Connected successfully!


In [4]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [5]:
models = client.models.list()

for model in models.data:
    print(f"Model ID: {model.id}")
    print(f"Owned By: {model.owned_by}")
    print(f"Created: {model.created}")
    print("-" * 40)

Model ID: openai/gpt-oss-20b
Owned By: OpenAI
Created: 1754407957
----------------------------------------
Model ID: whisper-large-v3
Owned By: OpenAI
Created: 1693721698
----------------------------------------
Model ID: openai/gpt-oss-safeguard-20b
Owned By: OpenAI
Created: 1761708789
----------------------------------------
Model ID: canopylabs/orpheus-arabic-saudi
Owned By: Canopy Labs
Created: 1765926439
----------------------------------------
Model ID: openai/gpt-oss-120b
Owned By: OpenAI
Created: 1754408224
----------------------------------------
Model ID: allam-2-7b
Owned By: SDAIA
Created: 1737672203
----------------------------------------
Model ID: meta-llama/llama-prompt-guard-2-22m
Owned By: Meta
Created: 1748632101
----------------------------------------
Model ID: qwen/qwen3.8-27b
Owned By: Alibaba Cloud
Created: 1786984846
----------------------------------------
Model ID: whisper-large-v3-turbo
Owned By: OpenAI
Created: 1728413088
------------------------------------

In [6]:
SCHEMA = """
DATABASE : crm_ai

TABLE: customers

customer_id INT PRIMARY KEY AUTO_INCREMENT
name VARCHAR(100)
email VARCHAR(100)
phone VARCHAR(20)
city VARCHAR(50)
state VARCHAR(50)
created_at DATETIME

TABLE: products

product_id INT PRIMARY KEY AUTO_INCREMENT
product_name VARCHAR(100)
category VARCHAR(50)
price DECIMAL(10,2)

TABLE: orders

order_id INT PRIMARY KEY AUTO_INCREMENT
customer_id INT FOREIGN KEY REFERENCES customers(customer_id)
order_date DATE
total_amount DECIMAL(10,2)
status VARCHAR(20)

TABLE: order_items

item_id INT PRIMARY KEY AUTO_INCREMENT
order_id INT FOREIGN KEY REFERENCES orders(order_id)
product_id INT FOREIGN KEY REFERENCES products(product_id)
quantity INT
amount DECIMAL(10,2)

TABLE: leads

lead_id INT PRIMARY KEY AUTO_INCREMENT
lead_name VARCHAR(100)
source VARCHAR(50)
status VARCHAR(30)
assigned_to VARCHAR(50)

TABLE: employees

employee_id INT PRIMARY KEY AUTO_INCREMENT
employee_name VARCHAR(100)
department VARCHAR(50)
email VARCHAR(100)

RELATIONSHIPS:

customers.customer_id
       |
       |
orders.customer_id

orders.order_id
       |
       |
order_items.order_id

products.product_id
       |
       |
order_items.product_id
"""

In [7]:
response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
        {
            "role": "user",
            "content": "Hello, Groq!"
        }
    ]
)
print(response.choices[0].message.content)

Hello! How can I assist you today?


### Dynamic Database Metadata Extraction

In [31]:
cursor.execute("""
SELECT
table_name,
column_name,
data_type
FROM information_schema.columns
WHERE table_schema='crm_ai'
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('customers', 'city', 'varchar')
('customers', 'created_at', 'datetime')
('customers', 'customer_id', 'int')
('customers', 'email', 'varchar')
('customers', 'name', 'varchar')
('customers', 'phone', 'varchar')
('customers', 'state', 'varchar')
('employees', 'department', 'varchar')
('employees', 'email', 'varchar')
('employees', 'employee_id', 'int')
('employees', 'employee_name', 'varchar')
('leads', 'assigned_to', 'varchar')
('leads', 'lead_id', 'int')
('leads', 'lead_name', 'varchar')
('leads', 'source', 'varchar')
('leads', 'status', 'varchar')
('order_items', 'amount', 'decimal')
('order_items', 'item_id', 'int')
('order_items', 'order_id', 'int')
('order_items', 'product_id', 'int')
('order_items', 'quantity', 'int')
('orders', 'customer_id', 'int')
('orders', 'order_date', 'date')
('orders', 'order_id', 'int')
('orders', 'status', 'varchar')
('orders', 'total_amount', 'decimal')
('products', 'category', 'varchar')
('products', 'price', 'decimal')
('products', 'product_id', 'int'

In [9]:
cursor.execute("""
SELECT
table_name,
column_name
FROM information_schema.key_column_usage
WHERE table_schema='crm_ai'
AND constraint_name='PRIMARY'
""")

pk_rows = cursor.fetchall()

print(pk_rows)

(('customers', 'customer_id'), ('employees', 'employee_id'), ('leads', 'lead_id'), ('order_items', 'item_id'), ('orders', 'order_id'), ('products', 'product_id'))


In [10]:
def get_schema():

    query = """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema='crm_ai'
    ORDER BY table_name, ordinal_position
    """

    cursor.execute(query)

    rows = cursor.fetchall()

    schema = ""

    current_table = None

    for table, column, dtype, nullable in rows:

        if current_table != table:

            schema += f"\n\nTABLE: {table}\n"

            current_table = table

        schema += (
            f"   {column} "
            f"({dtype}) "
            f"NULLABLE={nullable}\n"
        )

    return schema

In [11]:
def get_relationships():

    query = """
    SELECT
        table_name,
        column_name,
        referenced_table_name,
        referenced_column_name
    FROM information_schema.key_column_usage
    WHERE table_schema='crm_ai'
    AND referenced_table_name IS NOT NULL
    """

    cursor.execute(query)

    rows = cursor.fetchall()

    relationships = "\n\nRELATIONSHIPS:\n"

    for row in rows:

        relationships += (
            f"{row[0]}.{row[1]}"
            f" --> "
            f"{row[2]}.{row[3]}\n"
        )

    return relationships

In [12]:
def build_context():

    schema = get_schema()

    relationships = get_relationships()

    context = schema + relationships

    return context

In [13]:
build_context()

'\n\nTABLE: customers\n   customer_id (int) NULLABLE=NO\n   name (varchar) NULLABLE=YES\n   email (varchar) NULLABLE=YES\n   phone (varchar) NULLABLE=YES\n   city (varchar) NULLABLE=YES\n   state (varchar) NULLABLE=YES\n   created_at (datetime) NULLABLE=YES\n\n\nTABLE: employees\n   employee_id (int) NULLABLE=NO\n   employee_name (varchar) NULLABLE=YES\n   department (varchar) NULLABLE=YES\n   email (varchar) NULLABLE=YES\n\n\nTABLE: leads\n   lead_id (int) NULLABLE=NO\n   lead_name (varchar) NULLABLE=YES\n   source (varchar) NULLABLE=YES\n   status (varchar) NULLABLE=YES\n   assigned_to (varchar) NULLABLE=YES\n\n\nTABLE: order_items\n   item_id (int) NULLABLE=NO\n   order_id (int) NULLABLE=YES\n   product_id (int) NULLABLE=YES\n   quantity (int) NULLABLE=YES\n   amount (decimal) NULLABLE=YES\n\n\nTABLE: orders\n   order_id (int) NULLABLE=NO\n   customer_id (int) NULLABLE=YES\n   order_date (date) NULLABLE=YES\n   total_amount (decimal) NULLABLE=YES\n   status (varchar) NULLABLE=YES\n\

In [14]:
def generate_sql(user_question):
    schema_context = build_context()
    prompt = f"""
    You are an expert MySQL developer.
    DATABASE SCHEMA:
    {schema_context}
    RULES:
    1. Return ONLY executable MySQL.
    2. No markdown, no explanations, no comments.
    3. Use joins when required.
    4. Never generate DROP or TRUNCATE.
    USER QUESTION:
    {user_question}
    """
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    sql = response.choices[0].message.content.strip()

    sql = sql.replace("```sql", "").replace("```", "").strip()

    return sql

In [15]:
def validate_sql(sql):
    blocked_keywords = ["drop", "truncate"]
    sql_lower = sql.lower()
    for keyword in blocked_keywords:
        if keyword in sql_lower:
            raise Exception(f"Blocked Query Detected: {keyword}")
    return True

In [16]:
def explain_error(user_query, sql, error):
    prompt = f"""
    You are a CRM assistant.
    User Request: {user_query}
    Generated SQL: {sql}
    Database Error: {error}
    Explain the issue in simple business language and suggest what to do next.
    Do not show technical database jargon.
    """

    # Correct SDK call
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [17]:
def execute_sql_with_retry(sql, user_query, max_retries=2):
    """Executes SQL with an agentic self-correction loop if errors occur."""
    current_sql = sql
    
    for attempt in range(max_retries + 1):
        try:
            validate_sql(current_sql)
            cursor.execute(current_sql)
            
            if current_sql.lower().startswith("select"):
                rows = cursor.fetchall()
                columns = [d[0] for d in cursor.description]
                return pd.DataFrame(rows, columns=columns)
            
            conn.commit()
            return "Operation Successful"
            
        except Exception as e:
            error_msg = str(e)
            
            # If we've exhausted retries, fallback to friendly explanation
            if attempt == max_retries:
                return explain_error(user_query, current_sql, error_msg)
            
            # Self-Correction prompt to Gemini
            correction_prompt = f"""
            You are an expert MySQL developer fixing a broken query.
            
            Original User Request: {user_query}
            Failed SQL Query: {current_sql}
            MySQL Error: {error_msg}
            
            Please analyze the error, fix the SQL query, and return ONLY executable MySQL code without markdown or explanations.
            """
            
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {
                        "role": "user",
                        "content": correction_prompt
                    }
                ],
                temperature=0
            )

            current_sql = (
                response.choices[0]
                .message
                .content
                .replace("```sql", "")
                .replace("```", "")
                .strip()
            )

            print(
                f"Retrying with corrected SQL "
                f"(Attempt {attempt + 1}):\n{current_sql}"
            )

In [18]:
def classify_intent(user_query):
    """Classifies user input to ensure it falls within CRM boundaries."""
    prompt = f"""
    You are a security and intent router for a CRM AI assistant. 
    The database contains customers, products, orders, order items, leads, and employees.
    
    Analyze the user query: "{user_query}"
    
    Is this query related to querying, updating, or managing CRM data, business metrics, or entities?
    Answer ONLY with 'YES' or 'NO'.
    """
      
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content.strip().upper()

In [19]:
def auto_visualize(df, user_query):
    """Automatically detects data patterns and renders a Plotly chart if appropriate."""
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return
    
    # Check if DataFrame has at least one categorical/date column and one numeric column
    cat_cols = df.select_dtypes(include=['object', 'str', 'category', 'datetime']).columns
    num_cols = df.select_dtypes(include=['number']).columns
    
    if len(cat_cols) >= 1 and len(num_cols) >= 1:
        x_col = cat_cols[0]
        y_col = num_cols[0]
        
        # Avoid plotting IDs as metrics
        if 'id' not in x_col.lower() and len(df) > 1:
            print(f"\nRendering interactive chart for: {user_query}")
            
            # Use bar chart for categories, line chart if it looks like time-series
            if 'date' in x_col.lower() or 'created' in x_col.lower():
                fig = px.line(df, x=x_col, y=y_col, markers=True, title=f"Trend: {user_query}")
            else:
                fig = px.bar(df, x=x_col, y=y_col, text_auto=True, title=f"Insights: {user_query}")
                
            fig.update_layout(template="plotly_dark", xaxis_tickangle=-45)
            fig.show()

In [20]:
def ask_crm_secure(question):
    """Enhanced wrapper with intent checking and self-healing execution."""
    intent = classify_intent(question)
    
    if "NO" in intent:
        return "I'm your CRM assistant. I can only help you query, manage, or analyze customer, product, order, and lead data."
    
    sql = generate_sql(question)
    print("Generated SQL:\n", sql)
    
    result = execute_sql_with_retry(sql, question)
    return result

### Test Execution

In [21]:
query = "Show all customers from Kerala"
result = ask_crm_secure(query)

display(result)

Generated SQL:
 SELECT * FROM customers WHERE state = 'Kerala';


,customer_id,name,email,phone,city,state,created_at
0,1,Arun,arun@gmail.com,9876543210,Kochi,Kerala,2026-10-22 09:39:34
1,2,Rahul,rahul@gmail.com,9876543211,Kochi,Kerala,2026-10-22 09:39:34
2,3,Anu,anu@gmail.com,9876543212,Thrissur,Kerala,2026-10-22 09:39:34
3,4,Priya,priya@gmail.com,9876543213,Calicut,Kerala,2026-10-22 09:39:34
4,10,Vishnu,vishnu@gmail.com,9876543219,Kochi,Kerala,2026-10-22 09:39:34
5,11,Akhil,akhil@gmail.com,9876543220,Kollam,Kerala,2026-10-22 09:39:34
6,12,Meera,meera@gmail.com,9876543221,Kannur,Kerala,2026-10-22 09:39:34
7,13,Nikhil,nikhil@gmail.com,9876543222,Kochi,Kerala,2026-10-22 09:39:34
8,15,Hari,hari@gmail.com,9876543224,Palakkad,Kerala,2026-10-22 09:39:34
9,17,Kiran,kiran@gmail.com,9876543226,Kochi,Kerala,2026-10-22 09:39:34


In [22]:
auto_visualize(result, query)


Rendering interactive chart for: Show all customers from Kerala


In [23]:
ask_crm_secure("Delete table customers")

Generated SQL:
 DELETE FROM customers;
Retrying with corrected SQL (Attempt 1):
DELETE FROM orders;
DELETE FROM customers;
Retrying with corrected SQL (Attempt 2):
DROP TABLE IF EXISTS customers;


'**What happened:**  \nYou tried to remove the entire\u202f*Customers*\u202flist from the system. For safety reasons, the platform blocks that kind of action because it would erase all customer information at once.\n\n**What you can do next:**  \n\n1. **Remove only the records you don’t need** – If there are specific customers you want to get rid of, we can delete just those rows instead of the whole list.  \n2. **Archive the data first** – If you need to keep a copy of the current customers for reference, we can export the list to a file before deleting anything.  \n3. **Ask for special permission** – If you truly need to clear the entire table (for example, when setting up a brand‑new environment), you’ll need to request approval from a system administrator who can perform that step safely.\n\nLet me know which approach you’d like to take, and I’ll guide you through the next steps!'

In [24]:
result = ask_crm_secure(
    "Delete all lost leads"
)
display(result)

Generated SQL:
 DELETE FROM leads WHERE status = 'lost';


'Operation Successful'

In [25]:
result = ask_crm_secure(
    "Show top 5 customers by revenue"
)
display(result)

Generated SQL:
 SELECT c.customer_id, c.name, c.email, SUM(o.total_amount) AS revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.email
ORDER BY revenue DESC
LIMIT 5;


,customer_id,name,email,revenue
0,2,Rahul,rahul@gmail.com,95000.00
1,1,Arun,arun@gmail.com,75000.00
2,20,Anjali,anjali@gmail.com,45000.00
3,16,Neha,neha@gmail.com,35000.00
4,12,Meera,meera@gmail.com,30000.00


In [26]:
auto_visualize(result, "Show top 5 customers by revenue")


Rendering interactive chart for: Show top 5 customers by revenue


In [27]:
result = ask_crm_secure(
    "List customers who purchased Laptop Pro"
)
display(result)

Generated SQL:
 SELECT DISTINCT c.customer_id, c.name, c.email, c.phone, c.city, c.state
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE p.product_name = 'Laptop Pro';


,customer_id,name,email,phone,city,state
0,1,Arun,arun@gmail.com,9876543210,Kochi,Kerala


In [28]:
ask_crm_secure(
    "Show pending orders"
)


Generated SQL:
 SELECT o.order_id, c.name AS customer_name, o.order_date, o.total_amount, o.status
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'Pending';


,order_id,customer_name,order_date,total_amount,status
0,4,Priya,2026-06-04,3500.00,Pending
1,7,David,2026-06-07,800.00,Pending
2,11,Akhil,2026-06-11,25000.00,Pending
3,14,Ramesh,2026-06-14,2200.00,Pending
4,18,Deepa,2026-06-18,8500.00,Pending


In [29]:
ask_crm_secure(
    "add one row"
)

Generated SQL:
 INSERT INTO customers (customer_id, name, email, phone, city, state, created_at) VALUES (NULL, 'John Doe', 'john.doe@example.com', '555-1234', 'New York', 'NY', NOW());


'Operation Successful'